# Notebook d'analyse et de traitement des datasets


### Lecture des dataframes

Pour utiliser ce notebook il faut télécharger le document : StockEtablissement_utf8.csv présent à l'adresse suivante : https://static.data.gouv.fr/resources/base-sirene-des-entreprises-et-de-leurs-etablissements-siren-siret/20260701-092157/stock-stocketablissement-csv.zip (il est trop lourd pour être uploadé sur le github). Attention à bien le positionner au même niveau d'arborescence que ce notebook.  

In [251]:
import pandas as pd
import numpy as np


export_df = pd.read_csv('datasets example/export.csv', sep=';', dtype=str)
result_df = pd.read_csv('datasets example/result.csv', sep=';', dtype=str)
nomenclature_df = pd.read_csv('Nomenclature_ICPE.csv', sep=';', dtype=str)
columns_to_use = ['siret', 'activitePrincipaleEtablissement']
df_NAF = pd.read_csv('StockEtablissement_utf8.csv', usecols=columns_to_use, sep=',')

output_df = pd.read_csv("extraction/output_icpe_clean.csv", dtype={'entete_siret': str})
# Ajustement des types de données pour afficher le numéro correctement
output_df = output_df.dropna(subset=['situation_administrative_code_rubrique'])
num_etab = df["num_etablissement"].astype(str)
output_df['situation_administrative_code_rubrique'] = output_df['situation_administrative_code_rubrique'].astype(int)
output_df['num_etablissement'] = num_etab.str.rjust(10, '0')
output_df.to_csv("output_icpe_clean.csv", index=False)
output_df.head()


,num_etablissement,entete_nom,entete_siret,entete_etat_d_activite,entete_regime_en_vigueur_de_l_etablissement_2,entete_statut_seveso,entete_ied_-_mtd,derniere_date_inspection,situation_administrative_code_rubrique,situation_administrative_alinea,situation_administrative_libelle_rubrique,situation_administrative_regime_autorise,situation_administrative_volume
0,0000000039,COLLECTES VALORISATION ENERGIE DECHETS - COVED,34340353103542,En exploitation avec titre,Autorisation,Non Seveso,Oui,2026-01-26,2782,NaN,Autres traitements biologiques de déchets non ...,Autorisation,NaN
1,0000000039,COLLECTES VALORISATION ENERGIE DECHETS - COVED,34340353103542,En exploitation avec titre,Autorisation,Non Seveso,Oui,2026-01-26,3532,NaN,Valorisation de déchets non dangereux,Autorisation,518.000 t/j
2,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,42298971500186,En exploitation avec titre,Autorisation,Non Seveso,Non,NaN,1436,2,Liquides combustibles de point éclair compris ...,Déclaration avec contrôle,900.000 t
3,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,42298971500186,En exploitation avec titre,Autorisation,Non Seveso,Non,NaN,1450,1,Solides inflammables,Autorisation,150.000 t
4,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,42298971500186,En exploitation avec titre,Autorisation,Non Seveso,Non,NaN,1510,1,Entrepôts couverts soumis à EE systématique,Autorisation,NaN


## Analyse du datast result 

In [252]:
# On drop toutes les colonnes qui ne nous intéressent pas
result_df = result_df.drop(["bovins", "porcs", "volailles", "carriere", "eolienne", "industrie", "url_fiche", "code_epsg", "ied", "priorite_nationale", "num_dep", "seveso", "lib_seveso"], axis =1)


In [253]:
# On renomme les colonnes liées à l'ICPE
result_df = result_df.rename({"rubriques_autorisation" : "ICPE_A" , "rubriques_enregistrement" : "ICPE_E", "rubriques_declaration" : "ICPE_D"}, axis = 1)
result_df.columns


Index(['code_aiot', 'x', 'y', 'nom_ets', 'adresse', 'cd_insee', 'cd_postal',
       'commune', 'code_naf', 'lib_naf', 'num_siret', 'cd_regime',
       'lib_regime', 'ICPE_A', 'ICPE_E', 'ICPE_D', 'date_modification',
       'derniere_inspection'],
      dtype='object')

### Dédoublement des lignes par code ICPE

In [254]:
result_simple = result_df[result_df[['ICPE_A', 'ICPE_E', 'ICPE_D']].isna().all(axis=1)]

In [255]:
result_df["ICPE_A"] = result_df["ICPE_A"].apply( lambda x: f"{x}|0" if pd.notna(x) else x )
result_df["ICPE_E"] = result_df["ICPE_E"].apply( lambda x: f"{x}|0" if pd.notna(x) else x )

In [256]:
result_df['ICPE_A'] = result_df['ICPE_A'].str.split('|')
result_df = result_df.explode('ICPE_A')
result_df = result_df[result_df['ICPE_A'] != "nan"]

In [257]:
#On met des NaN là où il ne doit pas y avoir de valeurs
result_df.loc[((result_df['ICPE_A'] != "0") & (result_df["ICPE_A"].notna())), "ICPE_E"] = np.nan
result_df.loc[((result_df['ICPE_A'] != "0") & (result_df["ICPE_A"].notna())), "ICPE_D"] = np.nan


In [258]:
result_df['ICPE_E'] = result_df['ICPE_E'].str.split('|')
result_df = result_df.explode('ICPE_E')

In [259]:
result_df = result_df[result_df['ICPE_E'] != "nan"]

In [260]:
result_df.loc[((result_df['ICPE_E'] != "0") & (result_df["ICPE_E"] != "nan") & (result_df['ICPE_E'].notna() )), "ICPE_D"] = np.nan

In [261]:
result_df['ICPE_D'] = result_df['ICPE_D'].str.split('|')
result_df = result_df.explode('ICPE_D')

In [262]:
result_df = result_df[result_df['ICPE_D'] != "nan"]

In [263]:
result_df.loc[(result_df['ICPE_A'] == "0"), "ICPE_A"] = np.nan
result_df.loc[(result_df['ICPE_E'] == "0"), "ICPE_E"] = np.nan

In [264]:
result_df = result_df[~result_df[['ICPE_A', 'ICPE_E', 'ICPE_D']].isna().all(axis=1)]

In [265]:
result_trie = pd.concat([result_df, result_simple]).sort_values(by="code_aiot")

### On ajoute au fichier les descriptions des codes ICPE correspondantes

In [266]:
#On ne garde que les codes ICPE complets et non pas les grandes catégories
nomenclature_df = nomenclature_df[nomenclature_df['Service'].notna()]

In [267]:
nomenclature_df.head(10)

,Code,Description,Service
2,1185,Gaz à effet de serre fluorés,SSEEC-BPC
5,1312,Mise en œuvre de produits explosifs à des fins...,SRT-BRIEC
8,1413,Installations de remplissage de réservoirs de ...,SRT-BRIEC
9,1414,Installations de remplissage ou de distributio...,SRT-BRIEC
10,1416,Station services (hydrogène),SRT-BRIEC
12,1421,Installation de remplissage d'aérosols inflamm...,SRT-BRIEC
14,1434,Installations de remplissage ou de distributio...,SRT-BRIEC
15,1435,Stations-service,SRT-BRIEC
16,1436,Liquides de point éclair compris entre 60°C et...,SRT-BRIEC
18,1450,Solides inflammables,SRT-BRIEC


In [268]:
# On convertit les types pour la concaténation
nomenclature_df['Code'].astype(int)
result_trie['ICPE_A'].astype('Int64')

0         1450
0         <NA>
0         <NA>
0         <NA>
0         <NA>
          ... 
134851    <NA>
134852    <NA>
134853    <NA>
134854    <NA>
134855    <NA>
Name: ICPE_A, Length: 218723, dtype: Int64

In [269]:
result_trie['ICPE'] = result_trie['ICPE_A'].fillna('') + result_trie['ICPE_E'].fillna('') +  result_trie['ICPE_D'].fillna('')

In [270]:
# On merge
result = pd.merge(result_trie, nomenclature_df, left_on='ICPE', right_on="Code", how="left")


In [271]:
result=result.drop(columns=['ICPE', 'Code', 'Service'])

In [273]:
# On change l'ordre des colonnes pour plus de lisibilité
result= result[['code_aiot', 'nom_ets', 'x', 'y', 'adresse', 'cd_insee', 'cd_postal',
       'commune', 'code_naf', 'lib_naf', 'num_siret', 'cd_regime',
       'lib_regime', 'Description', 'ICPE_A', 'ICPE_E', 'ICPE_D',
       'date_modification', 'derniere_inspection']]
result= result.rename({'Description':'Description du code ICPE'}, axis = 1)
result.head()

,code_aiot,nom_ets,x,y,adresse,cd_insee,cd_postal,commune,code_naf,lib_naf,num_siret,cd_regime,lib_regime,Description du code ICPE,ICPE_A,ICPE_E,ICPE_D,date_modification,derniere_inspection
0,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,42298971500186,A,Autorisation,Solides inflammables,1450,NaN,NaN,2026/05/06 00:00:00,NaN
1,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,42298971500186,A,Autorisation,Alcools de bouche d'origine agricole,NaN,NaN,4755,2026/05/06 00:00:00,NaN
2,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,42298971500186,A,Autorisation,"Liquides comburants catégorie 1, 2 ou 3",NaN,NaN,4441,2026/05/06 00:00:00,NaN
3,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,42298971500186,A,Autorisation,Liquides inflammables de catégorie 2 ou catégo...,NaN,NaN,4331,2026/05/06 00:00:00,NaN
4,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,42298971500186,A,Autorisation,Liquides inflammables de catégorie 1,NaN,NaN,4330,2026/05/06 00:00:00,NaN


### On ajoute une information plus complète des codes NAF

In [274]:
df_NAF['siret']=df_NAF['siret'].astype(str)
result['num_siret'] = result['num_siret'].astype(str)

In [275]:
#On merge les dataframe sur le numéro de siret
result_NAF = pd.merge(result, df_NAF, left_on="num_siret", right_on="siret", how='left')

In [276]:
#Pour afficher les colonnes
with open('StockEtablissement_utf8.csv', 'r', encoding='utf-8') as f: 
    header = f.readline().strip().split(',')
    print(header)

['siren', 'nic', 'siret', 'statutDiffusionEtablissement', 'dateCreationEtablissement', 'trancheEffectifsEtablissement', 'anneeEffectifsEtablissement', 'activitePrincipaleRegistreMetiersEtablissement', 'dateDernierTraitementEtablissement', 'etablissementSiege', 'nombrePeriodesEtablissement', 'complementAdresseEtablissement', 'numeroVoieEtablissement', 'indiceRepetitionEtablissement', 'dernierNumeroVoieEtablissement', 'indiceRepetitionDernierNumeroVoieEtablissement', 'typeVoieEtablissement', 'libelleVoieEtablissement', 'codePostalEtablissement', 'libelleCommuneEtablissement', 'libelleCommuneEtrangerEtablissement', 'distributionSpecialeEtablissement', 'codeCommuneEtablissement', 'codeCedexEtablissement', 'libelleCedexEtablissement', 'codePaysEtrangerEtablissement', 'libellePaysEtrangerEtablissement', 'identifiantAdresseEtablissement', 'coordonneeLambertAbscisseEtablissement', 'coordonneeLambertOrdonneeEtablissement', 'complementAdresse2Etablissement', 'numeroVoie2Etablissement', 'indiceRe

#### Test de cohérence du code NAF du dataset StockEtablissement et celui du dataset result

In [277]:
result_NAF['indice_fiabilite_NAF']= result_NAF['activitePrincipaleEtablissement'].str[:2]
result_NAF['indice_fiabilite_NAF']=result_NAF['indice_fiabilite_NAF'].fillna(0)
result_NAF['code_naf']=result_NAF['code_naf'].fillna(0)
result_NAF['indice_fiabilite_NAF'].isna().sum()

np.int64(0)

In [278]:
result_NAF['indice_fiabilite_NAF']=result_NAF['indice_fiabilite_NAF'].astype(int)-result_NAF['code_naf'].astype(int)
result_NAF.head()

,code_aiot,nom_ets,x,y,adresse,cd_insee,cd_postal,commune,code_naf,lib_naf,...,lib_regime,Description du code ICPE,ICPE_A,ICPE_E,ICPE_D,date_modification,derniere_inspection,siret,activitePrincipaleEtablissement,indice_fiabilite_NAF
0,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,Autorisation,Solides inflammables,1450,NaN,NaN,2026/05/06 00:00:00,NaN,42298971500186,41.10A,0
1,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,Autorisation,Alcools de bouche d'origine agricole,NaN,NaN,4755,2026/05/06 00:00:00,NaN,42298971500186,41.10A,0
2,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,Autorisation,"Liquides comburants catégorie 1, 2 ou 3",NaN,NaN,4441,2026/05/06 00:00:00,NaN,42298971500186,41.10A,0
3,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,Autorisation,Liquides inflammables de catégorie 2 ou catégo...,NaN,NaN,4331,2026/05/06 00:00:00,NaN,42298971500186,41.10A,0
4,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,432849,6444403,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,Autorisation,Liquides inflammables de catégorie 1,NaN,NaN,4330,2026/05/06 00:00:00,NaN,42298971500186,41.10A,0


In [279]:
(result_NAF['indice_fiabilite_NAF'] == 0).sum(), result_NAF.shape

(np.int64(144617), (218723, 22))

## On récupère un fichier CSV propre à l'utilisation

In [280]:
# On récupère un csv trié
result_NAF.to_csv('result_NAF.csv', index=False)

# Création des bases de données finales

## Première base de donnée

In [330]:
df = pd.read_csv('result_NAF.csv', dtype={'num_siret': str})
df_naf = pd.read_csv('output_icpe_with_NAF.csv', dtype={'Siret': str, 'entete_siret': str, 
                                'coordonneeLambertOrdonneeEtablissement': str, 'coordonneeLambertAbscisseEtablissement': str})
df_naf.rename(columns={'entete_nom': 'Nom',
                   'entete_siret': 'Siret',
                   'situation_administrative_code_rubrique': 'Code_Rubrique',
                   'entete_statut_seveso': 'Statut_Seveso',
                   'situation_administrative_libelle_rubrique': 'Libelle_rubrique',
                   'situation_administrative_volume' : 'Volume'}, inplace=True)
df_naf = df_naf.drop(columns=['entete_etat_d_activite', 'entete_regime_en_vigueur_de_l_etablissement_2', 
                                            'entete_ied_-_mtd', 'situation_administrative_alinea'])

C:\Users\maxim\AppData\Local\Temp\ipykernel_25800\1587143361.py:1: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('result_NAF.csv', dtype={'num_siret': str})


In [331]:
# Nettoyage du csv pour faire un merge plus propre ultérieurement
df.rename(columns={'code_aiot': 'num_etablissement'}, inplace=True)
print(df.columns)

Index(['num_etablissement', 'nom_ets', 'x', 'y', 'adresse', 'cd_insee',
       'cd_postal', 'commune', 'code_naf', 'lib_naf', 'num_siret', 'cd_regime',
       'lib_regime', 'Description du code ICPE', 'ICPE_A', 'ICPE_E', 'ICPE_D',
       'date_modification', 'derniere_inspection', 'siret',
       'activitePrincipaleEtablissement', 'indice_fiabilite_NAF'],
      dtype='object')


In [332]:
num_etabli = df["num_etablissement"].astype(str)
df['num_etablissement'] = num_etabli.str.rjust(10, '0')
df.drop(columns=['x', 'y', 'date_modification', 'date_modification', 'indice_fiabilite_NAF'], inplace=True)
df = df.drop([ 'code_naf', 'lib_naf',  'cd_regime', 'lib_regime',
       'Description du code ICPE', 'ICPE_A', 'ICPE_E', 'ICPE_D',
       'derniere_inspection'], axis=1)

In [333]:
num_etab = df_naf["num_etablissement"].astype(str)
df_naf['num_etablissement'] = num_etab.str.rjust(10, '0')
df_naf = df_naf.drop(columns=[ 'Nom', 'Statut_Seveso',
        'Code_Rubrique', 'Libelle_rubrique',
       'situation_administrative_regime_autorise', 'Volume','Siret'])
df.columns


Index(['num_etablissement', 'nom_ets', 'adresse', 'cd_insee', 'cd_postal',
       'commune', 'num_siret', 'siret', 'activitePrincipaleEtablissement'],
      dtype='object')

In [334]:
df = df.merge(df_naf, on='num_etablissement', how='left')
df = df[['nom_ets', 'adresse', 'commune', 'cd_postal', 'cd_insee','coordonneeLambertAbscisseEtablissement',
       'coordonneeLambertOrdonneeEtablissement','num_siret','num_etablissement',
        'nomenclatureActivitePrincipaleEtablissement', 'derniere_date_inspection', 'activitePrincipaleEtablissement']]

df=df.drop_duplicates()
df.to_csv("dataset_final_1.csv")
df.head()

,nom_ets,adresse,commune,cd_postal,cd_insee,coordonneeLambertAbscisseEtablissement,coordonneeLambertOrdonneeEtablissement,num_siret,num_etablissement,nomenclatureActivitePrincipaleEtablissement,derniere_date_inspection,activitePrincipaleEtablissement
0,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,Gauriaguet,33240,33183.0,651515.2280706181,6863554.587773344,42298971500186,0000000002,NAFRev2,NaN,41.10A
380,SOCIETE FERROLAC,4 RUE PIERRE GILLES DE GENNES - ZAC DE LA VIGO...,Saint-Florent-sur-Cher,18400,18207.0,644020.72,6656809.55,78560962900064,0000000004,NAFRev2,2023-05-16,38.32Z
396,ORLEANS METROPOLE - Parc de Loire,5 PL DU 6 JUIN 1944 - ESPACE SAINT MARC - 4505...,Orléans,45000,45274.0,NaN,NaN,24450046800040,0000000005,NaN,NaN,84.11Z
397,GRANUPLAST FRANCE,754 rue de la liberté - ZI la grande borne,Jassans-Riottier,1480,1194.0,NaN,NaN,88325431000021,0000000010,NaN,NaN,38.32Z
398,EOLIEN - TOTAL QUADRAN CE La Luçoise,Forêt de Chaniaux - Lou Chambon,Luc,48250,48086.0,720377.3807893672,6251486.945676577,81519422000027,0000000034,NAFRev2,NaN,35.11Z


## Deuxième base de données

In [342]:
df2 = pd.read_csv("output_icpe_with_NAF.csv", dtype={'entete_siret': str, 'num_etablissement': str})
df_trie = pd.read_csv('result_NAF.csv', dtype={'num_siret': str, 'code_aiot': str})
df2 = df2.drop(columns=['entete_nom', 'entete_siret','entete_etat_d_activite',
       'entete_regime_en_vigueur_de_l_etablissement_2', 'entete_statut_seveso',
       'entete_ied_-_mtd', 'derniere_date_inspection',
       'situation_administrative_alinea',
       'situation_administrative_libelle_rubrique',
       'situation_administrative_regime_autorise', 'code_naf',
       'nomenclatureActivitePrincipaleEtablissement',
       'coordonneeLambertAbscisseEtablissement',
       'coordonneeLambertOrdonneeEtablissement'])
df_trie = df_trie.drop(columns=['nom_ets', 'x', 'y', 'adresse', 'cd_insee', 'cd_postal',
       'commune', 'code_naf', 'lib_naf', 'num_siret', 'cd_regime'])
df_trie[['ICPE_A','ICPE_E','ICPE_D']] = df_trie[['ICPE_A','ICPE_E','ICPE_D']].fillna(0)
df_trie[['ICPE_A','ICPE_E','ICPE_D']] = df_trie[['ICPE_A','ICPE_E','ICPE_D']].astype(int)
df_trie['ICPE'] = df_trie['ICPE_A'] + df_trie['ICPE_E'] + df_trie['ICPE_D']
print(df_trie.columns)
print(df2.columns)


C:\Users\maxim\AppData\Local\Temp\ipykernel_25800\526994232.py:2: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_trie = pd.read_csv('result_NAF.csv', dtype={'num_siret': str, 'code_aiot': str})


Index(['code_aiot', 'lib_regime', 'Description du code ICPE', 'ICPE_A',
       'ICPE_E', 'ICPE_D', 'date_modification', 'derniere_inspection', 'siret',
       'activitePrincipaleEtablissement', 'indice_fiabilite_NAF', 'ICPE'],
      dtype='object')
Index(['num_etablissement', 'situation_administrative_code_rubrique',
       'situation_administrative_volume'],
      dtype='object')


In [343]:
df_trie['code_aiot'] = df_trie['code_aiot'].astype(str)
df2['num_etablissement'] = df2['num_etablissement'].astype(str)
df_trie['ICPE'] = df_trie['ICPE'].astype(str)
df2['situation_administrative_code_rubrique'] = df2['situation_administrative_code_rubrique'].astype(str)
df_trie = df_trie.merge(df2, left_on=['code_aiot', 'ICPE'], right_on=['num_etablissement', 'situation_administrative_code_rubrique'])
df_trie = df_trie.drop(columns=['code_aiot'])
df_trie = df_trie.rename(columns={'code_aiot': 'num_etablissement', 'code_aiot': 'num_etablissement' })
num_etab = df_naf["num_etablissement"].astype(str)
df_naf['num_etablissement'] = num_etab.str.rjust(10, '0')
df_trie.head()

,lib_regime,Description du code ICPE,ICPE_A,ICPE_E,ICPE_D,date_modification,derniere_inspection,siret,activitePrincipaleEtablissement,indice_fiabilite_NAF,ICPE,num_etablissement,situation_administrative_code_rubrique,situation_administrative_volume
0,Autorisation,Solides inflammables,1450,0,0,2026/05/06 00:00:00,NaN,4.229897e+13,41.10A,0,1450,0000000002,1450,150.000 t
1,Autorisation,Alcools de bouche d'origine agricole,0,0,4755,2026/05/06 00:00:00,NaN,4.229897e+13,41.10A,0,4755,0000000002,4755,NaN
2,Autorisation,"Liquides comburants catégorie 1, 2 ou 3",0,0,4441,2026/05/06 00:00:00,NaN,4.229897e+13,41.10A,0,4441,0000000002,4441,4.000 t
3,Autorisation,Liquides inflammables de catégorie 2 ou catégo...,0,0,4331,2026/05/06 00:00:00,NaN,4.229897e+13,41.10A,0,4331,0000000002,4331,90.000 t
4,Autorisation,Liquides inflammables de catégorie 1,0,0,4330,2026/05/06 00:00:00,NaN,4.229897e+13,41.10A,0,4330,0000000002,4330,2.000 t


In [344]:
df_compo = pd.read_csv("reponse_claude/table1_correspondances.csv")
df_compo.head()

,Code_ICPE,Libelle_ICPE,Composant
0,1530,"Dépôts de papiers, cartons ou matériaux combus...",Papier
1,1530,"Dépôts de papiers, cartons ou matériaux combus...",Carton
2,1531,"Stockages, par voie humide (immersion ou asper...",Bois en fin de vie propre (palette)
3,1531,"Stockages, par voie humide (immersion ou asper...",Bois et sous-produits du bois
4,1532,Stockage de bois ou de matériaux combustibles ...,Bois en fin de vie propre (palette)


In [345]:
df_compo['Code_ICPE'] = df_compo['Code_ICPE'].astype(str)
df_trie = df_trie.merge(df_compo, left_on='ICPE', right_on='Code_ICPE', how='left')
df_trie = df_trie.drop(columns=['Libelle_ICPE','ICPE', 'Code_ICPE'])
df_trie = df_trie[['num_etablissement','lib_regime', 'ICPE_A', 'ICPE_E', 
                            'ICPE_D','Description du code ICPE', 'Composant',
                            'situation_administrative_volume','date_modification', 'derniere_inspection']]
df_trie.to_csv("dataset_final_2.csv")

In [346]:
df_trie.head()

,num_etablissement,lib_regime,ICPE_A,ICPE_E,ICPE_D,Description du code ICPE,Composant,situation_administrative_volume,date_modification,derniere_inspection
0,0000000002,Autorisation,1450,0,0,Solides inflammables,NaN,150.000 t,2026/05/06 00:00:00,NaN
1,0000000002,Autorisation,0,0,4755,Alcools de bouche d'origine agricole,NaN,NaN,2026/05/06 00:00:00,NaN
2,0000000002,Autorisation,0,0,4441,"Liquides comburants catégorie 1, 2 ou 3",NaN,4.000 t,2026/05/06 00:00:00,NaN
3,0000000002,Autorisation,0,0,4331,Liquides inflammables de catégorie 2 ou catégo...,NaN,90.000 t,2026/05/06 00:00:00,NaN
4,0000000002,Autorisation,0,0,4330,Liquides inflammables de catégorie 1,NaN,2.000 t,2026/05/06 00:00:00,NaN
